## Group Size Distribution Analysis technique

Group Size Distribution Analysis is an exploratory data analysis (EDA) technique used to determine how many records belong to each unique group and to summarize the distribution of those group sizes.

Code: df.groupby(col).size().value_counts().sort_index()

How it Works

groupby(col): Groups the dataset by a specific column.
Example: Group all rows belonging to the same heat_local_dict.

size(): Counts the number of rows in each group.

value_counts():Counts how many groups have each group size.

sort_index():Sorts the results by the group size in ascending order.

In [3]:
import pandas as pd

df = pd.DataFrame({
    "Heat": [
        "H1","H1",
        "H2","H2","H2",
        "H3",
        "H4","H4","H4",
        "H5","H5"
    ]
})

df

,Heat
0,H1
1,H1
2,H2
3,H2
4,H2
5,H3
6,H4
7,H4
8,H4
9,H5


In [3]:
## Step 1: Count rows in each group

df.groupby("Heat").size()

Heat
H1    2
H2    3
H3    1
H4    3
H5    2
dtype: int64

In [6]:
## Step 2: Distribution of group sizes

df.groupby("Heat").size().value_counts()

## 1 heat has 1 sample.  alt ek size ka group ek hi hai
## 2 heats have 2 samples. alt doh size ke groups are 2 in number
## 2 heats have 3 samples. alt 3 size ke groups are also 2 in number

2    2
3    2
1    1
Name: count, dtype: int64

In [5]:
df.groupby("Heat").size().value_counts().sort_index()

1    1
2    2
3    2
Name: count, dtype: int64

Applications:

Analyze the distribution of samples across groups.

Detect groups with very few or very many observations.

Identify imbalanced groups before model training.

Verify whether data collection is consistent across groups.

Decide whether a group-based train-test split (e.g., GroupShuffleSplit) is required.

Detect duplicate or repeated entities in the dataset.

Advantages:
Very fast and easy to compute.

Provides a clear summary of group frequencies.

Helps identify data quality issues.

Useful during exploratory data analysis (EDA).

## To know Total datatypes in columsn and their types


In [8]:
df=pd.DataFrame(
    {
        'name':['Mohit','Rohit','Ramesh'],
        'subject':['chem','phy','geo'],
        'marks':[15,50,95],
        'failed':[True,False,False]
    }
)

df.dtypes.value_counts()

object    2
int64     1
bool      1
Name: count, dtype: int64

## To Select Columns of particular DataType

In [13]:
cat_cols=df.select_dtypes(include='object')
num_df = df.select_dtypes(include=["int64", "float64"])
print(cat_cols)
print()
print(num_df)

     name subject
0   Mohit    chem
1   Rohit     phy
2  Ramesh     geo

   marks
0     15
1     50
2     95


## To know the outliers


In [2]:
import pandas as pd
df=pd.read_csv('/Users/priyanshugupta/Desktop/EDA Techniques/data.csv')

num_df=df.select_dtypes(include=['int32','float64'])
Q1=num_df.quantile(0.25)
Q3=num_df.quantile(0.75)
IQR=Q3-Q1
outlier_ratio=((num_df < Q1-1.5* IQR) | (num_df> Q3+1.5*IQR)).sum(axis=0)/len(df)
outliers=((num_df < (Q1 - 1.5*IQR)) | (num_df > (Q3 + 1.5*IQR))).sum()


## Dropping criteria:

1-Drop columns with missing data >=90 %

2-Drop categorical columns which is constant that is .nuinque() is <=1

3-Drop Duplicate Rows

4-Drop Cateogrical columns which have .nuinque() ratio >=0.98 that is almost all the value is different

5-Drop columns which have Variance < very small threshold that is ≥ 99% of values are the same

6-if 2 Columns are highly correlated, drop one of them

7-If a columns is not related to the target column at all

8-rows which have too many missing values 

In [ ]:
## 1-Drop columns with missing data >=90 %

missing_pct=df.isna().sum()/len(df)
cols_to_drop = missing_pct[missing_pct >= 0.9].index

In [39]:
## 2-Drop categorical columns which is constant that is .nuinque() is <=1
nunique = df.nunique()
constant_cols = nunique[nunique <= 1]

In [ ]:
## 3-Dropping duplicate rows
df.drop_duplicates()

,temperature,pressure,vibration,supplier,quality
0,27.0,103.9,6,B,0
1,31.8,98.1,18,A,1
2,24.1,101.2,30,A,1
3,34.8,99.8,14,A,1
4,34.6,102.4,33,B,0


In [48]:
## 4-Drop Cateogrical columns which have .nuinque() ratio >=0.98 that is almost all the value is differenthigh_unique_cols = df.columns[df.nunique() >= 0.98 * len(df)]
cat_cols=df.select_dtypes(include='object')
high_unique_cols = cat_cols.columns[cat_cols.nunique() >= 0.98 * len(df)]


In [4]:
## 5-Drop columns which have Variance < very small threshold that is ≥ 99% of values are the same

threshold = 0.99

quasi_constant_cols = []

numeric_df = df.select_dtypes(include=['number'])

for col in numeric_df.columns:
    # Get the frequency of the most common value
    predominant_value_count = df[col].value_counts(normalize=True).dropna().iloc[0]
    
    if predominant_value_count >= threshold:
        quasi_constant_cols.append(col)


In [3]:
## 6- Drop Highly correlated Columns
import numpy as np
corr_matrix = df.select_dtypes(include=[np.number]).corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] == 1.0)]

In [5]:
## 7-If a columns is not related to the target column at all

## lets say our target is quality
features = df.select_dtypes(include=['number']).columns
features = [f for f in features if f not in ['quality']]
corrs_p = df[features].corrwith(df['quality'], method='pearson').abs()
corrs_s=df[features].corrwith(df['quality'],method='spearman').abs()
weak_cols = corrs_p[(corrs_p<0.02) & (corrs_s<0.02)].index

In [6]:
## 8-remove rows with so many missing values

missing_per_row = df.isna().sum(axis=1)
df['percentage of missing data in a row'] = (missing_per_row / df.shape[1]) * 100
df = df[df['percentage of missing data in a row'] < 40].copy()